# 03. Catalogos y Torneo Fuzzy

## Goal
Load SCVS and SRI catalogs once, then run the fuzzy tournament to select the best candidate name or RUC for unmatched companies.


## Inputs
- Outputs from exact SCVS matching.
- `01_data_ingestion_enrichment/data_SRI/`
- `01_data_ingestion_enrichment/data_super_compañias/`

## Outputs
- `outputs/super_best.parquet`
- `outputs/sri_razon_best.parquet`
- `outputs/sri_fantasia_best.parquet`
- `outputs/leads_torneo_ganadores.csv`
- `outputs/horas_torneo_ganadores.csv`


### Configuración y Carga de Catálogos (SCVS y SRI)
Esta celda importa las librerías necesarias, configura las rutas del proyecto y define utilidades para leer/guardar archivos de forma estructurada. Su objetivo es asegurar que la lectura de los catálogos tanto de la SCVS como del SRI ocurra de forma segura e independiente del entorno de ejecución.

In [1]:

# ── Helpers y rutas ──────────────────────────────────────────────────────────
from pathlib import Path
import re, unicodedata, os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

try:
    from IPython.display import display
except Exception:
    def display(x): print(x)

def find_project_root(start=None):
    start = (start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "01_data_ingestion_enrichment").is_dir() and (p / "02_data_cleaning").is_dir():
            return p
    raise FileNotFoundError(f"No se pudo localizar la raíz. cwd: {start}")

def ensure_dir(d): Path(d).mkdir(parents=True, exist_ok=True); return Path(d)

def save_df_csv(df, path, *, index=False, encoding="utf-8-sig"):
    path = Path(path); ensure_dir(path.parent)
    df.to_csv(path, index=index, encoding=encoding)
    print(f"[OK] Guardado: {path.resolve()}  shape: {df.shape}")
    return path

def read_csv_checked(path, **kwargs):
    path = Path(path)
    if not path.exists(): raise FileNotFoundError(f"No existe: {path.resolve()}")
    return pd.read_csv(path, **kwargs)

ROOT         = find_project_root()
INGESTION_DIR = ROOT / "01_data_ingestion_enrichment"
CLEAN_DIR    = ROOT / "02_data_cleaning"
CLEAN_OUT    = CLEAN_DIR / "outputs"

# Buscar data_SRI en ambas ubicaciones posibles
_SRI_CANDIDATES = [CLEAN_DIR / "data_SRI", INGESTION_DIR / "data_SRI"]
SRI_DIR = next((p for p in _SRI_CANDIDATES if p.exists() and any(p.iterdir())), None)
if SRI_DIR is None:
    raise FileNotFoundError(f"No encontré data_SRI en ninguna de: {_SRI_CANDIDATES}")

# Buscar data_super_compañias en ambas ubicaciones posibles
_SCVS_CANDIDATES = [CLEAN_DIR / "data_super_compañias", INGESTION_DIR / "data_super_compañias"]
SCVS_DIR = next((p for p in _SCVS_CANDIDATES if p.exists()), None)
if SCVS_DIR is None:
    raise FileNotFoundError(f"No encontré data_super_compañias en ninguna de: {_SCVS_CANDIDATES}")

print(f"[CONFIG] ROOT     : {ROOT}")
print(f"[CONFIG] SRI_DIR  : {SRI_DIR}")
print(f"[CONFIG] SCVS_DIR : {SCVS_DIR}")
print(f"[CONFIG] CLEAN_OUT: {CLEAN_OUT}")


[CONFIG] ROOT     : E:\TESIS MAESTRIA\Desarrollo_clustering_maestria
[CONFIG] SRI_DIR  : E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\02_data_cleaning\data_SRI
[CONFIG] SCVS_DIR : E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\02_data_cleaning\data_super_compañias
[CONFIG] CLEAN_OUT: E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\02_data_cleaning\outputs


### Carga y Estandarización de Catálogos (SCVS y SRI)
En esta celda se centraliza la lectura y estandarización de las denominaciones de las empresas. Para ambos catálogos (SCVS y SRI) se definen funciones que buscan las columnas clave (RUC y razón social), limpian caracteres especiales y consolidan un listado unificado sin duplicados, el cual posteriormente de exporta en formato `pickle`.

In [2]:
# ── Funciones de normalización (self-contained) ───────────────────────────────

import pickle


def _strip_accents(s: str) -> str:
    return "".join(ch for ch in unicodedata.normalize("NFKD", s) if not unicodedata.combining(ch))


def normalize_company_name(value) -> str:
    if value is None: return ""
    if isinstance(value, float) and np.isnan(value): return ""
    s = str(value).strip()
    if not s: return ""
    s = re.sub(r"\[\s*\d+\s*\]", " ", s)
    s = re.sub(r"^\s*\d+\s*[\.)]\s*", " ", s)
    s = _strip_accents(s); s = s.upper()
    s = re.sub(r"[^A-Z0-9 ]+", " ", s)
    s = re.sub(r"\b(S\s*DE\s*R\s*L\s*DE\s*C\s*V|S\s*R\s*L\s*C\s*V|S\s*A\s*S|SAS|S\s*A|SA|C\s*LTDA|C\s*A|CA|C\s*V|LTDA|CIA|C\s*IA|COMPANIA|COMPAÑIA|ANONIMA|CORP|INC|LLC|C\s*L|\&|Y)\b", " ", s)
    s = re.sub(r"\b(DE|DEL|LA|EL|LOS|LAS)\b", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    tokens = []
    for tok in s.split():
        if not tokens or tokens[-1] != tok:
            tokens.append(tok)
    return " ".join(tokens)


def normalize_legal_exact_key(value) -> str:
    """Llave casi cruda: conserva designadores legales para exact match manual."""
    if value is None: return ""
    if isinstance(value, float) and np.isnan(value): return ""
    s = str(value).strip()
    if not s: return ""
    s = re.sub(r"\[\s*\d+\s*\]", " ", s)
    s = re.sub(r"^\s*\d+\s*[\.)]\s*", " ", s)
    s = _strip_accents(s).upper()
    s = s.replace("&", " Y ")
    s = re.sub(r"[^A-Z0-9]+", " ", s)
    s = re.sub(r"\b(COMPAÑIA|COMPANIA)\b", "COMPANIA", s)
    return re.sub(r"\s+", " ", s).strip()


def normalize_legal_compact_key(value) -> str:
    return re.sub(r"\s+", "", normalize_legal_exact_key(value))


def _normalize_ruc(value) -> str:
    if value is None: return ""
    if isinstance(value, float) and np.isnan(value): return ""
    s = re.sub(r"\D+", "", str(value).strip().replace(".0", ""))
    return s.zfill(13) if len(s) == 12 else s


def _valid_ruc_text(value) -> bool:
    return bool(re.fullmatch(r"\d{13}", _normalize_ruc(value)))


def _norm_col_key(col: str) -> str:
    s = str(col or "").strip().upper()
    s = "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))
    return re.sub(r"[^A-Z0-9]+", "", s)


def _unique_best_from_counts(counts: dict, key_col: str) -> pd.DataFrame:
    if not counts:
        return pd.DataFrame(columns=[key_col, "RUC", "freq"])
    df = pd.DataFrame([{key_col: k[0], "RUC": k[1], "freq": v} for k, v in counts.items()])
    df = df[(df[key_col].astype(str).str.len() >= 3) & (df["RUC"].astype(str).str.len() == 13)].copy()
    if df.empty:
        return pd.DataFrame(columns=[key_col, "RUC", "freq"])
    unique_keys = df.groupby(key_col)["RUC"].nunique()
    df = df[df[key_col].isin(unique_keys[unique_keys.eq(1)].index)].copy()
    return (
        df.sort_values([key_col, "freq"], ascending=[True, False])
          .drop_duplicates(key_col, keep="first")[[key_col, "RUC", "freq"]]
          .reset_index(drop=True)
    )


def _unique_best_from_frame(df: pd.DataFrame, key_col: str, ruc_col: str, out_ruc_col: str = "RUC") -> pd.DataFrame:
    if df.empty or key_col not in df.columns or ruc_col not in df.columns:
        return pd.DataFrame(columns=[key_col, out_ruc_col, "freq"])
    tmp = df[[key_col, ruc_col]].copy()
    tmp = tmp[(tmp[key_col].astype(str).str.len() >= 3) & (tmp[ruc_col].astype(str).str.len() == 13)]
    if tmp.empty:
        return pd.DataFrame(columns=[key_col, out_ruc_col, "freq"])
    counts = tmp.groupby([key_col, ruc_col]).size().reset_index(name="freq")
    unique_keys = counts.groupby(key_col)[ruc_col].nunique()
    counts = counts[counts[key_col].isin(unique_keys[unique_keys.eq(1)].index)].copy()
    counts = counts.rename(columns={ruc_col: out_ruc_col})
    return (
        counts.sort_values([key_col, "freq"], ascending=[True, False])
              .drop_duplicates(key_col, keep="first")[[key_col, out_ruc_col, "freq"]]
              .reset_index(drop=True)
    )


def _manual_reason_targets() -> tuple[set, set, set]:
    targets = []
    for fname in ["leads_ruc_exact.csv", "horas_ruc_exact.csv"]:
        p = CLEAN_OUT / fname
        if not p.exists():
            continue
        df = pd.read_csv(p, dtype={"RUC": "string"})
        ruc_ok = df.get("RUC", pd.Series("", index=df.index)).map(_normalize_ruc).astype(str).str.fullmatch(r"\d{13}")
        razon = df.get("razon_social_ecuador_raw", pd.Series("", index=df.index)).fillna("").astype(str).str.strip()
        validada = df.get("usa_razon_social_validada", pd.Series(False, index=df.index)).fillna(False).astype(str).str.upper().isin(["TRUE", "1", "1.0", "SI", "S", "YES", "Y"])
        targets.extend(razon[(~ruc_ok) & validada & razon.ne("")].tolist())
    legal = {normalize_legal_exact_key(x) for x in targets if normalize_legal_exact_key(x)}
    compact = {normalize_legal_compact_key(x) for x in targets if normalize_legal_compact_key(x)}
    norm = {normalize_company_name(x) for x in targets if normalize_company_name(x)}
    print(f"[MANUAL TARGETS] razones sociales pendientes: {len(targets)} | legal={len(legal)} compact={len(compact)} norm={len(norm)}")
    return legal, compact, norm


MANUAL_LEGAL_TARGETS, MANUAL_COMPACT_TARGETS, MANUAL_NORM_TARGETS = _manual_reason_targets()


# ── 1) Catálogo SCVS ──────────────────────────────────────────────────────────

def _pick_col(cols, patterns):
    cols_u = [str(c).upper() for c in cols]
    for pat in patterns:
        for i, cu in enumerate(cols_u):
            if pat in cu: return cols[i]
    return None


def _detect_excel_header_row(fp: Path, max_rows: int = 80) -> int:
    try: raw = pd.read_excel(fp, header=None, nrows=max_rows)
    except Exception: return 0
    best_i, best_score = 0, -1
    for i in range(len(raw)):
        vals = [v for v in raw.iloc[i].tolist() if str(v).lower() != "nan"]
        if not vals: continue
        rt = " ".join(map(str, vals)).upper()
        sc = sum(tok in rt for tok in ["RUC","IDENTIFIC"])*5 + sum(tok in rt for tok in ["NOMBRE","RAZON","RAZÓN"])*3
        if sc > best_score: best_score = sc; best_i = i
    return int(best_i)


def _load_super_companies(folder: Path):
    if not folder.exists(): raise FileNotFoundError(f"No existe la carpeta SCVS: {folder}")
    files = sorted([p for p in folder.iterdir() if p.suffix.lower() in {".xlsx",".xls",".csv"}])
    if not files: raise FileNotFoundError(f"Sin archivos en {folder}")
    frames = []
    for fp in files:
        try:
            df = pd.read_csv(fp, dtype=str) if fp.suffix.lower()==".csv" else pd.read_excel(fp, header=_detect_excel_header_row(fp), dtype=str)
        except Exception as e: print(f"[SCVS] {fp.name}: {e}"); continue
        if df is None or df.empty: continue
        df.columns = [str(c).strip() for c in df.columns]
        name_col = _pick_col(df.columns, ["RAZON","RAZÓN","NOMBRE","DENOM","COMPAÑ","EMPRESA"])
        ruc_col  = _pick_col(df.columns, ["RUC","IDENTIFIC","CEDULA"])
        if name_col is None:
            obj = [c for c in df.columns if str(df[c].dtype) in ("object","string")]
            name_col = obj[0] if obj else None
        if ruc_col is None:
            best, bsc = None, -1.0
            for c in df.columns:
                s = df[c].map(_normalize_ruc); sc = float((s!="").mean())+3*float((s.str.len()==13).mean())
                if sc>bsc: bsc=sc; best=c
            ruc_col = best
        if name_col is None or ruc_col is None: continue
        tmp = pd.DataFrame({"super_name_raw": df[name_col].astype("string").fillna("").map(str.strip), "super_ruc": df[ruc_col].map(_normalize_ruc)})
        tmp = tmp[tmp["super_ruc"].str.len()==13].copy()
        tmp["super_name_norm"] = tmp["super_name_raw"].map(normalize_company_name)
        tmp["super_legal_key"] = tmp["super_name_raw"].map(normalize_legal_exact_key)
        tmp["super_compact_key"] = tmp["super_name_raw"].map(normalize_legal_compact_key)
        tmp = tmp[tmp["super_name_norm"].str.contains(r"[A-Z]", regex=True, na=False) & (tmp["super_name_norm"].str.len()>=3)]
        if tmp.empty: continue
        frames.append(tmp[["super_name_raw", "super_name_norm", "super_legal_key", "super_compact_key", "super_ruc"]])
    if not frames: raise ValueError("No se pudo construir catálogo SCVS.")
    super_all = pd.concat(frames, ignore_index=True)
    super_best = (super_all.groupby(["super_name_norm","super_ruc"], as_index=False).size()
                  .sort_values(["super_name_norm","size"], ascending=[True,False])
                  .drop_duplicates("super_name_norm", keep="first")[["super_name_norm","super_ruc"]])
    super_norm_unique_best = _unique_best_from_frame(super_all, "super_name_norm", "super_ruc")
    super_legal_best = _unique_best_from_frame(super_all, "super_legal_key", "super_ruc")
    super_compact_best = _unique_best_from_frame(super_all, "super_compact_key", "super_ruc")
    return super_all, super_best, super_norm_unique_best, super_legal_best, super_compact_best


super_all, super_best, super_norm_unique_best, super_legal_best, super_compact_best = _load_super_companies(SCVS_DIR)
super_choices = super_best["super_name_norm"].dropna().tolist()
super_ruc_by_name = dict(zip(super_best["super_name_norm"], super_best["super_ruc"]))
super_ruc_by_name_unique = dict(zip(super_norm_unique_best["super_name_norm"], super_norm_unique_best["RUC"]))
super_ruc_by_legal_key = dict(zip(super_legal_best["super_legal_key"], super_legal_best["RUC"]))
super_ruc_by_compact_key = dict(zip(super_compact_best["super_compact_key"], super_compact_best["RUC"]))
print(f"[SCVS] distinct nombres: {len(super_choices)}")
print(f"[SCVS] norm keys únicas: {len(super_norm_unique_best)}  exact legal: {len(super_legal_best)}  compact: {len(super_compact_best)}")

ensure_dir(CLEAN_OUT)
with open(CLEAN_OUT / "super_best.pkl", "wb") as f: pickle.dump(super_best, f)
with open(CLEAN_OUT / "super_norm_unique_best.pkl", "wb") as f: pickle.dump(super_norm_unique_best, f)
with open(CLEAN_OUT / "super_legal_best.pkl", "wb") as f: pickle.dump(super_legal_best, f)
with open(CLEAN_OUT / "super_compact_best.pkl", "wb") as f: pickle.dump(super_compact_best, f)
print(f"[OK] super_best.pkl y llaves exactas SCVS guardados")


# ── 2) Catálogos SRI ──────────────────────────────────────────────────────────
SRI_CHUNK_ROWS = int(os.getenv("SRI_CHUNK_ROWS","200000"))
USE_CACHED_SRI_CATALOGS = os.getenv("USE_CACHED_SRI_CATALOGS", "1").strip() != "0"


def _sniff_sep(fp: Path) -> str:
    try: head = fp.open("rb").read(4096)
    except Exception: return ","
    counts = {"|": head.count(b"|"), ";": head.count(b";"), ",": head.count(b","), "\t": head.count(b"\t")}
    sep = max(counts, key=counts.get)
    return sep if counts.get(sep, 0) > 0 else ","


def _pick_sri_columns(fp: Path):
    sep = _sniff_sep(fp)
    for enc in ["utf-8-sig","utf-8","cp1252","latin1"]:
        try:
            hdr = pd.read_csv(fp, sep=sep, encoding=enc, dtype=str, nrows=0, on_bad_lines="skip")
            norm_to_orig = {_norm_col_key(c): c for c in hdr.columns}
            ruc_col  = next((norm_to_orig[a] for a in ["NUMERORUC","NUMERODERUC","RUC"] if a in norm_to_orig), None)
            razon_col= next((norm_to_orig[a] for a in ["RAZONSOCIAL","RAZONSOC"] if a in norm_to_orig), None)
            fan_col  = next((norm_to_orig[a] for a in ["NOMBREFANTASIACOMERCIAL","NOMBREFANTASIA","NOMBRECOMERCIAL"] if a in norm_to_orig), None)
            if ruc_col and razon_col:
                return sep, enc, ruc_col, razon_col, fan_col
        except Exception: continue
    raise RuntimeError(f"No detecté columnas SRI en {fp.name}")


def _load_pickle_if_exists(path: Path):
    if path.exists():
        with open(path, "rb") as f:
            return pickle.load(f)
    return None


def _load_or_build_sri_catalogs(sri_dir: Path):
    razon_p = CLEAN_OUT / "sri_razon_best.pkl"
    fantasia_p = CLEAN_OUT / "sri_fantasia_best.pkl"
    if USE_CACHED_SRI_CATALOGS and razon_p.exists() and fantasia_p.exists():
        print("[SRI] Usando catálogos normalizados cacheados para torneo")
        return _load_pickle_if_exists(razon_p), _load_pickle_if_exists(fantasia_p)

    files = sorted([p for p in sri_dir.iterdir() if p.suffix.lower()==".csv"])
    if not files: raise FileNotFoundError(f"Sin CSV en {sri_dir}")
    count_razon, count_fan = {}, {}
    for i, fp in enumerate(files, 1):
        print(f"[SRI/FULL] ({i}/{len(files)}) {fp.name}")
        try: sep, enc, col_ruc, col_razon, col_fan = _pick_sri_columns(fp)
        except Exception as e: print(f"  Saltado: {e}"); continue
        usecols = [c for c in [col_ruc, col_razon, col_fan] if c]
        for chunk in pd.read_csv(fp, sep=sep, encoding=enc, dtype=str, usecols=usecols,
                                 chunksize=SRI_CHUNK_ROWS, low_memory=True, on_bad_lines="skip"):
            rename = {col_ruc:"NUMERO_RUC", col_razon:"RAZON_SOCIAL"}
            if col_fan: rename[col_fan] = "NOMBRE_FANTASIA_COMERCIAL"
            chunk = chunk.rename(columns=rename)
            if "NOMBRE_FANTASIA_COMERCIAL" not in chunk.columns:
                chunk["NOMBRE_FANTASIA_COMERCIAL"] = pd.NA
            chunk["NUMERO_RUC"] = chunk["NUMERO_RUC"].map(_normalize_ruc)
            tmp = chunk[["NUMERO_RUC","RAZON_SOCIAL"]].copy()
            tmp = tmp[tmp["RAZON_SOCIAL"].astype("string").notna()]
            tmp = tmp[tmp["RAZON_SOCIAL"].astype("string").str.strip() != ""]
            if not tmp.empty:
                tmp["name_norm"] = tmp["RAZON_SOCIAL"].map(normalize_company_name)
                tmp = tmp[(tmp["NUMERO_RUC"].str.len()==13) & (tmp["name_norm"].str.len()>=3)]
                for (nn, ruc), cnt in tmp.groupby(["name_norm","NUMERO_RUC"]).size().items():
                    count_razon[(str(nn), str(ruc))] = count_razon.get((str(nn), str(ruc)), 0) + int(cnt)
            tmpf = chunk[["NUMERO_RUC","NOMBRE_FANTASIA_COMERCIAL"]].copy()
            tmpf = tmpf[tmpf["NOMBRE_FANTASIA_COMERCIAL"].astype("string").notna()]
            tmpf = tmpf[tmpf["NOMBRE_FANTASIA_COMERCIAL"].astype("string").str.strip() != ""]
            if not tmpf.empty:
                tmpf["fan_norm"] = tmpf["NOMBRE_FANTASIA_COMERCIAL"].map(normalize_company_name)
                tmpf = tmpf[(tmpf["NUMERO_RUC"].str.len()==13) & (tmpf["fan_norm"].str.len()>=3)]
                for (fn, ruc), cnt in tmpf.groupby(["fan_norm","NUMERO_RUC"]).size().items():
                    count_fan[(str(fn), str(ruc))] = count_fan.get((str(fn), str(ruc)), 0) + int(cnt)
    sri_razon_best = (
        pd.DataFrame([{"sri_razon_norm": k[0], "RUC": k[1], "freq": v} for k,v in count_razon.items()])
        .sort_values(["sri_razon_norm","freq"], ascending=[True,False])
        .drop_duplicates("sri_razon_norm", keep="first")[["sri_razon_norm","RUC"]]
        .reset_index(drop=True)
    )
    if count_fan:
        sri_fantasia_best = (
            pd.DataFrame([{"sri_fantasia_norm": k[0], "RUC": k[1], "freq": v} for k,v in count_fan.items()])
            .sort_values(["sri_fantasia_norm","freq"], ascending=[True,False])
            .drop_duplicates("sri_fantasia_norm", keep="first")[["sri_fantasia_norm","RUC"]]
            .reset_index(drop=True)
        )
    else:
        sri_fantasia_best = pd.DataFrame(columns=["sri_fantasia_norm","RUC"])
    return sri_razon_best, sri_fantasia_best


def _scan_sri_for_manual_targets(sri_dir: Path):
    files = sorted([p for p in sri_dir.iterdir() if p.suffix.lower()==".csv"])
    count_norm, count_legal, count_compact = {}, {}, {}
    if not (MANUAL_LEGAL_TARGETS or MANUAL_COMPACT_TARGETS or MANUAL_NORM_TARGETS):
        return (
            pd.DataFrame(columns=["sri_razon_norm", "RUC", "freq"]),
            pd.DataFrame(columns=["sri_razon_legal_key", "RUC", "freq"]),
            pd.DataFrame(columns=["sri_razon_compact_key", "RUC", "freq"]),
        )
    for i, fp in enumerate(files, 1):
        print(f"[SRI/MANUAL] ({i}/{len(files)}) {fp.name}")
        try: sep, enc, col_ruc, col_razon, _ = _pick_sri_columns(fp)
        except Exception as e: print(f"  Saltado: {e}"); continue
        for chunk in pd.read_csv(fp, sep=sep, encoding=enc, dtype=str, usecols=[col_ruc, col_razon],
                                 chunksize=SRI_CHUNK_ROWS, low_memory=True, on_bad_lines="skip"):
            chunk = chunk.rename(columns={col_ruc:"NUMERO_RUC", col_razon:"RAZON_SOCIAL"})
            chunk["NUMERO_RUC"] = chunk["NUMERO_RUC"].map(_normalize_ruc)
            chunk = chunk[chunk["NUMERO_RUC"].str.len()==13].copy()
            if chunk.empty: continue
            chunk["legal_key"] = chunk["RAZON_SOCIAL"].map(normalize_legal_exact_key)
            chunk["compact_key"] = chunk["RAZON_SOCIAL"].map(normalize_legal_compact_key)
            chunk["name_norm"] = chunk["RAZON_SOCIAL"].map(normalize_company_name)
            m = chunk[chunk["legal_key"].isin(MANUAL_LEGAL_TARGETS)]
            for (kk, ruc), cnt in m.groupby(["legal_key", "NUMERO_RUC"]).size().items():
                count_legal[(str(kk), str(ruc))] = count_legal.get((str(kk), str(ruc)), 0) + int(cnt)
            m = chunk[chunk["compact_key"].isin(MANUAL_COMPACT_TARGETS)]
            for (kk, ruc), cnt in m.groupby(["compact_key", "NUMERO_RUC"]).size().items():
                count_compact[(str(kk), str(ruc))] = count_compact.get((str(kk), str(ruc)), 0) + int(cnt)
            m = chunk[chunk["name_norm"].isin(MANUAL_NORM_TARGETS)]
            for (kk, ruc), cnt in m.groupby(["name_norm", "NUMERO_RUC"]).size().items():
                count_norm[(str(kk), str(ruc))] = count_norm.get((str(kk), str(ruc)), 0) + int(cnt)
    return (
        _unique_best_from_counts(count_norm, "sri_razon_norm"),
        _unique_best_from_counts(count_legal, "sri_razon_legal_key"),
        _unique_best_from_counts(count_compact, "sri_razon_compact_key"),
    )


sri_razon_best, sri_fantasia_best = _load_or_build_sri_catalogs(SRI_DIR)
sri_razon_unique_best, sri_razon_legal_best, sri_razon_compact_best = _scan_sri_for_manual_targets(SRI_DIR)

sri_razon_choices = sri_razon_best["sri_razon_norm"].dropna().tolist()
sri_razon_ruc_by_name = dict(zip(sri_razon_best["sri_razon_norm"], sri_razon_best["RUC"]))
sri_razon_ruc_by_name_unique = dict(zip(sri_razon_unique_best["sri_razon_norm"], sri_razon_unique_best["RUC"]))
sri_razon_ruc_by_legal_key = dict(zip(sri_razon_legal_best["sri_razon_legal_key"], sri_razon_legal_best["RUC"]))
sri_razon_ruc_by_compact_key = dict(zip(sri_razon_compact_best["sri_razon_compact_key"], sri_razon_compact_best["RUC"]))
sri_fan_choices = sri_fantasia_best["sri_fantasia_norm"].dropna().tolist()
sri_fan_ruc_by_name = dict(zip(sri_fantasia_best["sri_fantasia_norm"], sri_fantasia_best["RUC"]))

with open(CLEAN_OUT / "sri_razon_best.pkl", "wb") as f: pickle.dump(sri_razon_best, f)
with open(CLEAN_OUT / "sri_fantasia_best.pkl", "wb") as f: pickle.dump(sri_fantasia_best, f)
with open(CLEAN_OUT / "sri_razon_unique_best.pkl", "wb") as f: pickle.dump(sri_razon_unique_best, f)
with open(CLEAN_OUT / "sri_razon_legal_best.pkl", "wb") as f: pickle.dump(sri_razon_legal_best, f)
with open(CLEAN_OUT / "sri_razon_compact_best.pkl", "wb") as f: pickle.dump(sri_razon_compact_best, f)
print(f"[SRI] Razón distinct: {len(sri_razon_best)}  Fantasía distinct: {len(sri_fantasia_best)}")
print(f"[SRI] Matches manuales únicos: norm={len(sri_razon_unique_best)} legal={len(sri_razon_legal_best)} compact={len(sri_razon_compact_best)}")
print(f"[OK] sri_razon_best.pkl, sri_fantasia_best.pkl y llaves objetivo SRI guardados")


[MANUAL TARGETS] razones sociales pendientes: 71 | legal=71 compact=71 norm=71


[SCVS] distinct nombres: 214261
[SCVS] norm keys únicas: 214063  exact legal: 214379  compact: 214257


[OK] super_best.pkl y llaves exactas SCVS guardados
[SRI] Usando catálogos normalizados cacheados para torneo


[SRI/MANUAL] (1/26) SRI_Catastro_Empresas_Fantasmas.csv
[SRI/MANUAL] (2/26) SRI_MERCADOSENLINEA.csv
[SRI/MANUAL] (3/26) SRI_RUC_Azuay.csv


[SRI/MANUAL] (4/26) SRI_RUC_Bolivar.csv


...

[SRI/MANUAL] (23/26) SRI_RUC_Santo_Domingo.csv


[SRI/MANUAL] (24/26) SRI_RUC_Sucumbios.csv


[SRI/MANUAL] (25/26) SRI_RUC_Tungurahua.csv


[SRI/MANUAL] (26/26) SRI_RUC_Zamora_Chinchipe.csv


[SRI] Razón distinct: 6646760  Fantasía distinct: 1466058
[SRI] Matches manuales únicos: norm=23 legal=22 compact=23
[OK] sri_razon_best.pkl, sri_fantasia_best.pkl y llaves objetivo SRI guardados


## Migrate Here From Source Notebook
- Centralized SRI and SCVS catalog loading.
- Leads fuzzy tournament.
- Horas fuzzy tournament.


### Torneo de Similitud Fuzzy
Aplica un cruce masivo mediante `rapidfuzz` (o `difflib`) entre empresas no clasificadas en el paso anterior y los catálogos consolidados. Busca coincidencia superior a un umbral (por defecto 80%), prioriza qué origen usar si hay empates (SCVS tiene más prioridad que SRI) y documenta a los "ganadores" en sus correspondientes archivos CSV.

In [3]:
# ── Match determinístico manual + Torneo Fuzzy de huérfanos ───────────────────

leads_ruc_exact = read_csv_checked(CLEAN_OUT / "leads_ruc_exact.csv", dtype={"RUC": "string"})
horas_ruc_exact  = read_csv_checked(CLEAN_OUT / "horas_ruc_exact.csv", dtype={"RUC": "string"})

UMBRAL = int(os.getenv("UMBRAL_SRI", "80"))
INCLUDE_SRI_FANTASIA = os.getenv("INCLUDE_SRI_FANTASIA", "0").strip() == "1"
_PRIO  = {"SCVS": 0, "SRI_RAZON": 1, "SRI_FANTASIA": 2}

try:
    from rapidfuzz import process as _rfp, fuzz as _rff
    _HAVE_RF = True
    print("[TORNEO] Motor: rapidfuzz (batch cdist, workers=-1)")
except ImportError:
    _HAVE_RF = False
    print("[TORNEO] Motor: difflib (fallback secuencial — instala rapidfuzz para mayor velocidad)")


def _is_valid_ruc_value(value) -> bool:
    return bool(re.fullmatch(r"\d{13}", _normalize_ruc(value)))


def _is_valid_ruc_series(s: pd.Series) -> pd.Series:
    return s.map(_normalize_ruc).astype(str).str.fullmatch(r"\d{13}")


def _bool_series(s: pd.Series) -> pd.Series:
    return s.fillna(False).astype(str).str.strip().str.upper().isin(["TRUE", "1", "1.0", "SI", "S", "YES", "Y"])


def _nonempty_series(s: pd.Series) -> pd.Series:
    return s.fillna("").astype(str).str.strip().ne("")


def _split_manual_vs_orphans(df: pd.DataFrame, *, raw_col: str, norm_col: str, orig_norm_col: str, label: str):
    d = df.copy()
    ruc_ok = _is_valid_ruc_series(d["RUC"]) if "RUC" in d.columns else pd.Series(False, index=d.index)
    razon_llena = _nonempty_series(d["razon_social_ecuador_raw"]) if "razon_social_ecuador_raw" in d.columns else pd.Series(False, index=d.index)
    razon_validada = _bool_series(d["usa_razon_social_validada"]) if "usa_razon_social_validada" in d.columns else pd.Series(False, index=d.index)

    sin_ruc = ~ruc_ok
    tiene_razon_manual_ec = razon_llena & razon_validada

    manual_sin_ruc = d[sin_ruc & tiene_razon_manual_ec].copy()
    huerfanos = d[sin_ruc & ~tiene_razon_manual_ec].copy()

    # Para huérfanos se consulta por la primera columna original, no por razón social Ecuador.
    if orig_norm_col in huerfanos.columns:
        huerfanos[norm_col] = huerfanos[orig_norm_col].fillna(huerfanos[norm_col]).astype(str).str.strip()

    print(f"[{label}] Sin RUC exacto: {int(sin_ruc.sum())}")
    print(f"[{label}] Manual con razón social validada pero sin RUC: {len(manual_sin_ruc)} -> match determinístico, no torneo")
    print(f"[{label}] Huérfanos para torneo con primera columna: {len(huerfanos)}")
    return manual_sin_ruc, huerfanos


def _lookup_manual_razon(razon_raw: str) -> dict:
    legal_key = normalize_legal_exact_key(razon_raw)
    compact_key = normalize_legal_compact_key(razon_raw)
    norm_key = normalize_company_name(razon_raw)
    attempts = [
        ("MANUAL_RAZON_SCVS_LEGAL", legal_key, super_ruc_by_legal_key),
        ("MANUAL_RAZON_SRI_LEGAL", legal_key, sri_razon_ruc_by_legal_key),
        ("MANUAL_RAZON_SCVS_COMPACT", compact_key, super_ruc_by_compact_key),
        ("MANUAL_RAZON_SRI_COMPACT", compact_key, sri_razon_ruc_by_compact_key),
        ("MANUAL_RAZON_SCVS_NORM_UNIQUE", norm_key, super_ruc_by_name_unique),
        ("MANUAL_RAZON_SRI_NORM_UNIQUE", norm_key, sri_razon_ruc_by_name_unique),
    ]
    hits = []
    for method, key, mapping in attempts:
        ruc = _normalize_ruc(mapping.get(key, "")) if key else ""
        if _is_valid_ruc_value(ruc):
            hits.append({"method": method, "key": key, "RUC": ruc})
    rucs = sorted({h["RUC"] for h in hits})
    if len(rucs) == 1:
        first = hits[0]
        return {
            "status": "MATCHED",
            "RUC": rucs[0],
            "method": first["method"],
            "key": first["key"],
            "hits": "; ".join(f"{h['method']}={h['RUC']}" for h in hits),
        }
    if len(rucs) > 1:
        return {
            "status": "CONFLICTO_RUC",
            "RUC": "",
            "method": "",
            "key": legal_key,
            "hits": "; ".join(f"{h['method']}={h['RUC']}" for h in hits),
        }
    return {"status": "SIN_RUC_DETERMINISTICO", "RUC": "", "method": "", "key": legal_key, "hits": ""}


def _match_manual_razon_exact(manual_df: pd.DataFrame, *, label: str, raw_col: str, norm_col: str):
    if manual_df.empty:
        return manual_df.copy(), manual_df.copy()
    d = manual_df.copy()
    lookups = d["razon_social_ecuador_raw"].map(_lookup_manual_razon)
    lookup_df = pd.DataFrame(list(lookups), index=d.index)
    d["manual_match_status"] = lookup_df["status"]
    d["manual_match_method"] = lookup_df["method"]
    d["manual_match_key"] = lookup_df["key"]
    d["manual_match_hits"] = lookup_df["hits"]
    d["RUC"] = lookup_df["RUC"].map(_normalize_ruc)

    matched = d[d["manual_match_status"].eq("MATCHED") & d["RUC"].map(_is_valid_ruc_value)].copy()
    review = d[~d.index.isin(matched.index)].copy()

    if not matched.empty:
        matched["source_label"] = label
        matched["source_winner"] = matched["manual_match_method"]
        matched["winner_name_norm"] = matched["razon_social_ecuador_raw"].map(normalize_company_name)
        matched["score"] = 100.0
    print(f"[{label}] Razón social manual -> RUC determinístico: {len(matched)}")
    if not review.empty:
        print(review["manual_match_status"].value_counts(dropna=False).to_string())
    return matched, review


def _run_tournament(unmatched: pd.DataFrame, q_col: str, raw_col: str) -> pd.DataFrame:
    COLS = [raw_col, q_col, "source_winner", "winner_name_norm", "score", "RUC"]

    queries  = unmatched[q_col].fillna("").astype(str).str.strip().tolist()
    raw_vals = unmatched[raw_col].tolist() if raw_col in unmatched.columns else [""] * len(queries)
    if not any(queries):
        return pd.DataFrame(columns=COLS)

    catalogs = [
        (super_choices,       super_ruc_by_name,     "SCVS"),
        (sri_razon_choices,   sri_razon_ruc_by_name, "SRI_RAZON"),
    ]
    if INCLUDE_SRI_FANTASIA:
        catalogs.append((sri_fan_choices, sri_fan_ruc_by_name, "SRI_FANTASIA"))
    print("[TORNEO] Catálogos usados:", ", ".join(src for _, _, src in catalogs))

    n = len(queries)
    best_scores = np.full(n, -1.0, dtype=np.float32)
    best_idxs   = np.full(n, -1,   dtype=np.int32)
    best_srcs   = [""] * n

    if _HAVE_RF:
        # Modo rápido: cdist calcula toda la matriz en C con paralelismo real
        CHUNK = 500_000  # bloques para no saturar RAM con catálogos muy grandes
        for choices, _, src in catalogs:
            if not choices: continue
            for start in range(0, len(choices), CHUNK):
                sub = choices[start:start + CHUNK]
                mat = _rfp.cdist(queries, sub,
                                 scorer=_rff.token_set_ratio,
                                 score_cutoff=UMBRAL - 1,
                                 workers=-1,
                                 dtype=np.float32)
                sub_idx  = mat.argmax(axis=1)
                sub_sc   = mat[np.arange(n), sub_idx]
                sub_glob = np.array(sub_idx + start, dtype=np.int32)

                improve = (sub_sc >= UMBRAL) & (
                    (sub_sc > best_scores) |
                    ((sub_sc == best_scores) &
                     np.array([_PRIO[src] < _PRIO.get(best_srcs[i], 99) for i in range(n)]))
                )
                best_scores = np.where(improve, sub_sc,   best_scores)
                best_idxs   = np.where(improve, sub_glob, best_idxs)
                best_srcs   = [src if improve[i] else best_srcs[i] for i in range(n)]
    else:
        # Fallback difflib
        from difflib import SequenceMatcher
        for choices, _, src in catalogs:
            if not choices: continue
            for i, q in enumerate(queries):
                if not q: continue
                best_sc, best_j = -1.0, -1
                for j, c in enumerate(choices):
                    sc = SequenceMatcher(None, q, c).ratio() * 100
                    if sc > best_sc: best_sc, best_j = sc, j
                if best_sc >= UMBRAL and (best_sc > best_scores[i] or
                   (best_sc == best_scores[i] and _PRIO[src] < _PRIO.get(best_srcs[i], 99))):
                    best_scores[i] = best_sc
                    best_idxs[i]   = best_j
                    best_srcs[i]   = src

    ruc_maps  = {"SCVS": super_ruc_by_name, "SRI_RAZON": sri_razon_ruc_by_name, "SRI_FANTASIA": sri_fan_ruc_by_name}
    cat_lists = {"SCVS": super_choices,     "SRI_RAZON": sri_razon_choices,     "SRI_FANTASIA": sri_fan_choices}

    rows = []
    for i in range(n):
        q = queries[i]
        if not q or best_scores[i] < UMBRAL: continue
        src_win   = best_srcs[i]
        best_name = cat_lists[src_win][best_idxs[i]]
        ruc       = ruc_maps[src_win].get(best_name, "")
        rows.append({raw_col: raw_vals[i], q_col: q, "source_winner": src_win,
                     "winner_name_norm": best_name, "score": float(best_scores[i]),
                     "RUC": str(ruc) if ruc else ""})

    if not rows:
        return pd.DataFrame(columns=COLS)
    return pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)


leads_manual_sin_ruc, unmatched_leads = _split_manual_vs_orphans(
    leads_ruc_exact,
    raw_col="Company_raw",
    norm_col="Company_norm",
    orig_norm_col="Company_norm_original",
    label="LEADS",
)
horas_manual_sin_ruc, unmatched_horas = _split_manual_vs_orphans(
    horas_ruc_exact,
    raw_col="EMPRESA_raw",
    norm_col="EMPRESA_norm",
    orig_norm_col="EMPRESA_norm_original",
    label="HORAS",
)

leads_manual_ruc_exact, leads_manual_sin_ruc_revision = _match_manual_razon_exact(
    leads_manual_sin_ruc, label="LEADS", raw_col="Company_raw", norm_col="Company_norm"
)
horas_manual_ruc_exact, horas_manual_sin_ruc_revision = _match_manual_razon_exact(
    horas_manual_sin_ruc, label="HORAS", raw_col="EMPRESA_raw", norm_col="EMPRESA_norm"
)

leads_torneo_ganadores = _run_tournament(unmatched_leads, "Company_norm", "Company_raw")
horas_torneo_ganadores  = _run_tournament(unmatched_horas,  "EMPRESA_norm",  "EMPRESA_raw")

print(f"\n[LEADS/TORNEO] Ganadores huérfanos: {len(leads_torneo_ganadores)}")
if not leads_torneo_ganadores.empty:
    print(leads_torneo_ganadores["source_winner"].value_counts().to_string())
print(f"\n[HORAS/TORNEO] Ganadores huérfanos: {len(horas_torneo_ganadores)}")
if not horas_torneo_ganadores.empty:
    print(horas_torneo_ganadores["source_winner"].value_counts().to_string())

_ = save_df_csv(leads_manual_ruc_exact, CLEAN_OUT / "leads_manual_razon_ruc_exact.csv")
_ = save_df_csv(horas_manual_ruc_exact,  CLEAN_OUT / "horas_manual_razon_ruc_exact.csv")
_ = save_df_csv(leads_manual_sin_ruc_revision, CLEAN_OUT / "leads_manual_sin_ruc_revision.csv")
_ = save_df_csv(horas_manual_sin_ruc_revision,  CLEAN_OUT / "horas_manual_sin_ruc_revision.csv")
_ = save_df_csv(leads_torneo_ganadores, CLEAN_OUT / "leads_torneo_ganadores.csv")
_ = save_df_csv(horas_torneo_ganadores,  CLEAN_OUT / "horas_torneo_ganadores.csv")

display(leads_manual_ruc_exact.head(10))
display(leads_torneo_ganadores.head(10))


[TORNEO] Motor: rapidfuzz (batch cdist, workers=-1)
[LEADS] Sin RUC exacto: 241
[LEADS] Manual con razón social validada pero sin RUC: 41 -> match determinístico, no torneo
[LEADS] Huérfanos para torneo con primera columna: 200
[HORAS] Sin RUC exacto: 51
[HORAS] Manual con razón social validada pero sin RUC: 30 -> match determinístico, no torneo
[HORAS] Huérfanos para torneo con primera columna: 21
[LEADS] Razón social manual -> RUC determinístico: 11
manual_match_status
SIN_RUC_DETERMINISTICO    30
[HORAS] Razón social manual -> RUC determinístico: 14
manual_match_status
SIN_RUC_DETERMINISTICO    16
[TORNEO] Catálogos usados: SCVS, SRI_RAZON


[TORNEO] Catálogos usados: SCVS, SRI_RAZON



[LEADS/TORNEO] Ganadores huérfanos: 158
source_winner
SRI_RAZON    89
SCVS         69

[HORAS/TORNEO] Ganadores huérfanos: 19
source_winner
SRI_RAZON    11
SCVS          8
[OK] Guardado: E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\02_data_cleaning\outputs\leads_manual_razon_ruc_exact.csv  shape: (11, 20)
[OK] Guardado: E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\02_data_cleaning\outputs\horas_manual_razon_ruc_exact.csv  shape: (14, 20)
[OK] Guardado: E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\02_data_cleaning\outputs\leads_manual_sin_ruc_revision.csv  shape: (30, 16)
[OK] Guardado: E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\02_data_cleaning\outputs\horas_manual_sin_ruc_revision.csv  shape: (16, 16)
[OK] Guardado: E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\02_data_cleaning\outputs\leads_torneo_ganadores.csv  shape: (158, 6)
[OK] Guardado: E:\TESIS MAESTRIA\Desarrollo_clustering_maestria\02_data_cleaning\outputs\horas_torneo_ganadores.csv  shape: (19, 6)


,Company_raw,Company_norm_original,original_raw_validacion,razon_social_ecuador_raw,pais_validacion,sede_ecuador,validacion_manual,razon_social_ecuador_norm,usa_razon_social_validada,Company_match_raw,Company_norm,RUC,manual_match_status,manual_match_method,manual_match_key,manual_match_hits,source_label,source_winner,winner_name_norm,score
2,Abbvie,ABBVIE,Abbvie,ABBVIE S.A.S.,EEUU,True,True,ABBVIE,True,ABBVIE S.A.S.,ABBVIE,1792489156001,MATCHED,MANUAL_RAZON_SRI_LEGAL,ABBVIE S A S,MANUAL_RAZON_SRI_LEGAL=1792489156001; MANUAL_R...,LEADS,MANUAL_RAZON_SRI_LEGAL,ABBVIE,100.0
19,APOTEX,APOTEX,APOTEX,APOTEX ECUADOR S.A.,Canadá,True,True,APOTEX ECUADOR,True,APOTEX ECUADOR S.A.,APOTEX ECUADOR,1791241762001,MATCHED,MANUAL_RAZON_SRI_LEGAL,APOTEX ECUADOR S A,MANUAL_RAZON_SRI_LEGAL=1791241762001; MANUAL_R...,LEADS,MANUAL_RAZON_SRI_LEGAL,APOTEX ECUADOR,100.0
23,TECNOLOGICO DE MONTERREY,TECNOLOGICO MONTERREY,TECNOLOGICO DE MONTERREY,ASOCIACION CIVIL DEL INSTITUTO TECNOLOGICO Y D...,México,True,True,ASOCIACION CIVIL INSTITUTO TECNOLOGICO ESTUDIO...,True,ASOCIACION CIVIL DEL INSTITUTO TECNOLOGICO Y D...,ASOCIACION CIVIL INSTITUTO TECNOLOGICO ESTUDIO...,0991463712001,MATCHED,MANUAL_RAZON_SRI_LEGAL,ASOCIACION CIVIL DEL INSTITUTO TECNOLOGICO Y D...,MANUAL_RAZON_SRI_LEGAL=0991463712001; MANUAL_R...,LEADS,MANUAL_RAZON_SRI_LEGAL,ASOCIACION CIVIL INSTITUTO TECNOLOGICO ESTUDIO...,100.0
41,BRINSA S.A.,BRINSA,BRINSA S.A.,BRINSA S.A.S..,Colombia,True,True,BRINSA,True,BRINSA S.A.S..,BRINSA,0991349073001,MATCHED,MANUAL_RAZON_SRI_NORM_UNIQUE,BRINSA,MANUAL_RAZON_SRI_NORM_UNIQUE=0991349073001,LEADS,MANUAL_RAZON_SRI_NORM_UNIQUE,BRINSA,100.0
91,Econofarm S.A Corporación GPF,ECONOFARM CORPORACION GPF,Econofarm S.A Corporación GPF,ECONOFARM S.A.,Ecuador,True,True,ECONOFARM,True,ECONOFARM S.A.,ECONOFARM,1791715772001,MATCHED,MANUAL_RAZON_SRI_LEGAL,ECONOFARM S A,MANUAL_RAZON_SRI_LEGAL=1791715772001; MANUAL_R...,LEADS,MANUAL_RAZON_SRI_LEGAL,ECONOFARM,100.0
176,ICONN,ICONN,ICONN,Icon S.A.S.,México,True,True,ICON,True,Icon S.A.S.,ICON,1793194826001,MATCHED,MANUAL_RAZON_SRI_LEGAL,ICON S A S,MANUAL_RAZON_SRI_LEGAL=1793194826001; MANUAL_R...,LEADS,MANUAL_RAZON_SRI_LEGAL,ICON,100.0
198,Maver,MAVER,Maver,LABORATORIOS MAVER DEL ECUADOR S.A.,Chile,True,True,LABORATORIOS MAVER ECUADOR,True,LABORATORIOS MAVER DEL ECUADOR S.A.,LABORATORIOS MAVER ECUADOR,1792047986001,MATCHED,MANUAL_RAZON_SRI_LEGAL,LABORATORIOS MAVER DEL ECUADOR S A,MANUAL_RAZON_SRI_LEGAL=1792047986001; MANUAL_R...,LEADS,MANUAL_RAZON_SRI_LEGAL,LABORATORIOS MAVER ECUADOR,100.0
208,LITCORP,LITCORP,LITCORP,LITCORP S.A.,Ecuador,True,True,LITCORP,True,LITCORP S.A.,LITCORP,0991370609001,MATCHED,MANUAL_RAZON_SRI_LEGAL,LITCORP S A,MANUAL_RAZON_SRI_LEGAL=0991370609001; MANUAL_R...,LEADS,MANUAL_RAZON_SRI_LEGAL,LITCORP,100.0
221,METLIFE MÉXICO,METLIFE MEXICO,METLIFE MÉXICO,"MetLife, Inc",México,True,True,METLIFE,True,"MetLife, Inc",METLIFE,1792916178001,MATCHED,MANUAL_RAZON_SRI_NORM_UNIQUE,METLIFE,MANUAL_RAZON_SRI_NORM_UNIQUE=1792916178001,LEADS,MANUAL_RAZON_SRI_NORM_UNIQUE,METLIFE,100.0
251,PHILLIPS,PHILLIPS,PHILLIPS,PHILIPS ECUADOR CA,Paises bajos,True,True,PHILIPS ECUADOR,True,PHILIPS ECUADOR CA,PHILIPS ECUADOR,1790018156001,MATCHED,MANUAL_RAZON_SRI_LEGAL,PHILIPS ECUADOR CA,MANUAL_RAZON_SRI_LEGAL=1790018156001; MANUAL_R...,LEADS,MANUAL_RAZON_SRI_LEGAL,PHILIPS ECUADOR,100.0


,Company_raw,Company_norm,source_winner,winner_name_norm,score,RUC
0,ACCO BRANDS,ACCO BRANDS,SRI_RAZON,BRANDS,100.0,0991336753001
1,"ACH FOOD COMPANIES, INC",ACH FOOD COMPANIES,SRI_RAZON,FOOD,100.0,1792752310001
2,ALLIANZ MÉXICO,ALLIANZ MEXICO,SCVS,ALLIANZ,100.0,0195120420001
3,AFP Genesis,AFP GENESIS,SCVS,AFP GENESIS ADMINISTRADORA FONDOS FIDEICOMISOS,100.0,0991307605001
4,AVERY DENNISON RBIS,AVERY DENNISON RBIS,SRI_RAZON,AVERY,100.0,0991367209001
5,La Anita,ANITA,SCVS,COMERCIAL IMPORTADORA SANTA ANITA IMSANIT,100.0,0990085188001
6,AXIONLOG,AXIONLOG,SCVS,AXIONLOG ECUADOR,100.0,0992991178001
7,BPL BIO PRODUCTS LABORATORY,BPL BIO PRODUCTS LABORATORY,SRI_RAZON,LABORATORY,100.0,1793196784001
8,CFB,CFB,SCVS,CENTRO FERRETERO BAMBOO CFB,100.0,1793230370001
9,COMEX,COMEX,SCVS,ASOCIADOS EN SOLUCIONES EMPRESARIALES COMEX AS...,100.0,0993290599001


### Verificación de Salidas
Esta celda de cierre evalúa que todos los archivos requeridos creados a lo largo del proceso del *notebook* (entre ellos los diccionarios en `pickle` y los torneos ganados en CSV) se encuentren disponibles físicamente en las rutas esperadas en el disco y muestra que las filas contengan la forma correcta.

In [4]:

# ── Verificación final ────────────────────────────────────────────────────────
import pickle

required = [
    CLEAN_OUT / "super_best.pkl",
    CLEAN_OUT / "super_norm_unique_best.pkl",
    CLEAN_OUT / "super_legal_best.pkl",
    CLEAN_OUT / "super_compact_best.pkl",
    CLEAN_OUT / "sri_razon_best.pkl",
    CLEAN_OUT / "sri_razon_unique_best.pkl",
    CLEAN_OUT / "sri_razon_legal_best.pkl",
    CLEAN_OUT / "sri_razon_compact_best.pkl",
    CLEAN_OUT / "sri_fantasia_best.pkl",
    CLEAN_OUT / "leads_manual_razon_ruc_exact.csv",
    CLEAN_OUT / "horas_manual_razon_ruc_exact.csv",
    CLEAN_OUT / "leads_manual_sin_ruc_revision.csv",
    CLEAN_OUT / "horas_manual_sin_ruc_revision.csv",
    CLEAN_OUT / "leads_torneo_ganadores.csv",
    CLEAN_OUT / "horas_torneo_ganadores.csv",
]
for p in required:
    if not p.exists(): raise FileNotFoundError(f"Output faltante: {p}")
    if p.suffix == ".pkl":
        with open(p, "rb") as f: df_chk = pickle.load(f)
    else:
        df_chk = pd.read_csv(p)
    print(f"[OK] {p.name:45s}  shape={df_chk.shape}")
print("\n✓ Notebook 03_catalogos_y_torneo_fuzzy completado correctamente.")


[OK] super_best.pkl                                 shape=(214261, 2)
[OK] super_norm_unique_best.pkl                     shape=(214063, 3)


[OK] super_legal_best.pkl                           shape=(214379, 3)
[OK] super_compact_best.pkl                         shape=(214257, 3)


[OK] sri_razon_best.pkl                             shape=(6646760, 2)


[OK] sri_razon_unique_best.pkl                      shape=(23, 3)
[OK] sri_razon_legal_best.pkl                       shape=(22, 3)
[OK] sri_razon_compact_best.pkl                     shape=(23, 3)


[OK] sri_fantasia_best.pkl                          shape=(1466058, 2)
[OK] leads_manual_razon_ruc_exact.csv               shape=(11, 20)
[OK] horas_manual_razon_ruc_exact.csv               shape=(14, 20)
[OK] leads_manual_sin_ruc_revision.csv              shape=(30, 16)
[OK] horas_manual_sin_ruc_revision.csv              shape=(16, 16)
[OK] leads_torneo_ganadores.csv                     shape=(158, 6)


[OK] horas_torneo_ganadores.csv                     shape=(19, 6)

✓ Notebook 03_catalogos_y_torneo_fuzzy completado correctamente.
